# Matched Gemma base and IT experiments
Enable the existing HF_TOKEN secret; select two T4 GPUs and internet; Save & Run All. Runs both checkpoints sequentially with the same FP16 native-hook backend, 24 held-out word-pair TV cells plus 9 sentiment TV cells per model (66 total). Technical pilots gate test evaluation; accuracy never gates inclusion. Results and source hashes are saved in outputs; weights and credentials are not. See execution_protocol.md in outputs.

In [ ]:
import os,json,time,zlib,base64,sys,hashlib,datetime,importlib.metadata,traceback
from pathlib import Path
os.environ['HF_HOME']='/tmp/hf-cache'
# Preserve the environment validated by the completed bridge notebook.
import subprocess
subprocess.run([sys.executable,'-m','pip','install','--quiet','transformers==5.0.0','accelerate==1.13.0'],check=True)
import torch
from kaggle_secrets import UserSecretsClient
files=json.loads(zlib.decompress(base64.b64decode('...')))
OUT=Path('/kaggle/working');ROOT=OUT/'source'
for name,content in files.items():
 p=ROOT/name;p.parent.mkdir(parents=True,exist_ok=True);p.write_text(content)
sys.path.insert(0,str(ROOT/'src'));sys.path.insert(0,str(ROOT/'scripts'))
from fvtv import kaggle_backend as backend,eval_icl,tv,fv,tasks,heldout
import run_heldout,run_sentiment_mapping
from types import SimpleNamespace
from run_heldout import write_json
backend.TOKEN=UserSecretsClient().get_secret('HF_TOKEN')
backend.REVISIONS=json.loads(files['revisions.json']);backend.OUTPUT=OUT
backend.DEADLINE=min(time.time()+11.5*3600,datetime.datetime.fromisoformat('2026-09-08T20:00:00+02:00').timestamp())
assert torch.cuda.device_count()==2, 'Expected two T4 GPUs'
for m in [run_heldout,run_sentiment_mapping]:m.load_model=backend.load_model
eval_icl.predict_top1=backend.predict_top1
tv.extract_theta_all_layers=backend.extract_theta_all_layers
tv.patch_theta=backend.patch_theta
fv.verify_arch=backend.verify_arch
execution={'started':time.time(),'dtype':'float16','batch_size':4,'revisions':backend.REVISIONS,'deadline':backend.DEADLINE,'versions':{p:importlib.metadata.version(p) for p in ['torch','transformers','accelerate']},'source_hashes':{name:hashlib.sha256(content.encode()).hexdigest() for name,content in files.items()},'models':{}}
write_json(OUT/'execution.json',execution)

def pilot(model):
 backend.verify_arch(model)
 split=heldout.partition(tasks.load_task('antonym'))
 demos=tasks.sample_demos(split['construction'],100)
 dummy=tv.pick_dummy_query(split['construction'],demos,655)
 vectors=backend.extract_theta_all_layers(model,demos,dummy)
 prompt=tasks.build_icl_prompt(demos,split['dev'][0]['input'])
 own=backend.extract_theta_all_layers(model,demos,split['dev'][0]['input'],prompt=prompt)
 clean=backend.predict_top1(model,[prompt])
 for L in [0,20,41]:assert backend.patch_theta(model,[prompt],own[L],L)==clean, 'Identity check failed'
 prompts=[tasks.build_zeroshot_prompt(p['input']) for p in split['dev'][:4]]
 assert backend.patch_theta(model,prompts,vectors[20],20,batch_size=1)==backend.patch_theta(model,prompts,vectors[20],20,batch_size=4), 'Batching mismatch'
 return {'passed':True,'identity_layers':[0,20,41],'batching_inputs':4}

for model in ['gemma-2-9b','gemma-2-9b-it']:
 state=execution['models'][model]={'stage':'loading','complete':False}
 write_json(OUT/'execution.json',execution)
 try:
  m=backend.load_model('google/'+model)
  state['pilot']=pilot(m)
  del m
  state['stage']='word-pair';write_json(OUT/'execution.json',execution)
  run_heldout.run(SimpleNamespace(model=model,method='tv',tasks=','.join(heldout.TASK_NAMES),seeds='100,101,102',out=str(OUT/'results/heldout'/model/'tv')))
  state['stage']='sentiment-pilot';write_json(OUT/'execution.json',execution)
  args=SimpleNamespace(model=model,data=str(ROOT/'sentiment.json'),out=str(OUT/'results/sentiment'/model),phase='pilot')
  run_sentiment_mapping.run(args)
  state['stage']='sentiment-full';write_json(OUT/'execution.json',execution)
  args.phase='full';run_sentiment_mapping.run(args)
  state.update(stage='complete',complete=True,finished=time.time())
 except Exception as exc:
  state.update(stage='failed',error_type=type(exc).__name__)
  # Source frame locations aid diagnosis without printing credentials or local variables.
  state['error_frames']=[{'file':Path(f.filename).name,'line':f.lineno,'function':f.name} for f in traceback.extract_tb(exc.__traceback__)]
  print(model,'STOPPED',type(exc).__name__,flush=True)
 m=None
 write_json(OUT/'execution.json',execution)
print('EXPERIMENT FINISHED',json.dumps(execution['models']),flush=True)
